# D1 精讲：两数之和（LeetCode 1）

> **先说一句：这题你现在不会做，完全正常。**
> D1 的目标是建立「套路」，不是证明自己聪明。看完这篇，你今晚遇到同类题就有手感了。

---

## 一、题目

给定一个**整数数组** `nums` 和一个**整数目标值** `target`，
请你在该数组中找出**和为目标值 target 的那两个整数**，并返回它们的**数组下标**。

假设每种输入只会对应一个答案，且**同一个元素不能使用两次**。

**例子：**

```
nums = [2, 7, 11, 15],  target = 9
答案 = [0, 1]
因为 nums[0] + nums[1] = 2 + 7 = 9
```


## 二、先不问「最优解」，先问「最笨的办法」

面试时最强的推法不是背答案，而是：**先说最笨的办法，再想怎么优化。**

最笨的办法：把所有两两组合都试一遍。


In [1]:
def two_sum_brute(nums, target):
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):        # j 从 i+1 开始，保证同一个元素不被用两次
            print(f"试 {nums[i]} + {nums[j]} = {nums[i] + nums[j]}")
            if nums[i] + nums[j] == target:
                return [i, j]
    return []


print("结果:", two_sum_brute([2, 7, 11, 15], 9))

试 2 + 7 = 9
结果: [0, 1]


## 三、暴力解的问题：重复劳动

`n = 4` 时比较 6 次，看起来没什么。但增长是这样的：

| n | 比较次数 |
|---|---|
| 4 | 6 |
| 1,000 | 约 50 万 |
| 100,000 | 约 50 亿 → **超时** |

时间复杂度是 **O(n²)**。

**停下来想一件事**：这个循环的核心动作是什么？

> 对每个数 x，我在找 `target - x` 在不在数组里。

问题不在「找什么」，而在**「怎么找」**——我们每次都在从头扫一遍数组。
有没有一种东西，**查得特别快**？


## 四、工具：哈希表（Python 里就是 dict）

`dict` 就是一种哈希表：`{键: 值}`。
它的关键特性是——**查一个键几乎是瞬间的，不管字典里存了多少项**（平均 O(1)）。

对比一下 `list` 和 `dict` 的查找速度：

In [2]:
import time

n = 200_000
big_list = list(range(n))
big_dict = {v: i for i, v in enumerate(big_list)}   # {数字: 下标}

目标 = 999_999        # 故意找一个不存在的值（最坏情况：list 必须扫完）

t0 = time.perf_counter(); _ = 目标 in big_list; t1 = time.perf_counter()
t2 = time.perf_counter(); _ = 目标 in big_dict; t3 = time.perf_counter()

print(f"list 查找耗时: {(t1 - t0) * 1000:.3f} ms")
print(f"dict 查找耗时: {(t3 - t2) * 1000:.3f} ms")
print(f"快了多少倍: {(t1 - t0) / max(t3 - t2, 1e-9):,.0f} 倍")

list 查找耗时: 1.982 ms
dict 查找耗时: 0.024 ms
快了多少倍: 82 倍


看到差距了吗？这就是「用空间换时间」——**多用一个字典，把 O(n²) 变成 O(n)**。

小实验：字典怎么用来查「搭档」

In [3]:
# 假设我已经见过 2（下标0）和 7（下标1）
seen = {2: 0, 7: 1}          # 键是数字，值是这个数字的下标

当前数字 = 7
需要找的搭档 = 9 - 当前数字

print("搭档是:", 需要找的搭档)
print("搭档在字典里吗?", 需要找的搭档 in seen)
print("搭档的下标是:", seen[需要找的搭档])
print("一个没见过的键呢?", 100 in seen)

搭档是: 2
搭档在字典里吗? True
搭档的下标是: 0
一个没见过的键呢? False


## 五、最优解：一遍扫描

思路只有两句话：

1. 遍历每个数 `x`，先问：**「我需要的搭档 `target - x`，之前在字典里见过吗？」**
   - **见过** → 立刻返回 `[搭档的下标, 当前下标]`
   - **没见过** → 把当前的 `x` 和它的下标记进字典，供后面的数来找

2. 注意顺序：**必须先查、后存**。
   如果先存再查，`target = 2x` 时（比如 `nums=[3,3], target=6`）你会把同一个元素用两次，结果是错的。

复杂度：时间 **O(n)**，空间 **O(n)**。

In [4]:
def two_sum(nums, target):
    seen = {}                            # 键: 已经见过的数字；值: 它的下标
    for i, x in enumerate(nums):
        need = target - x                # 我需要的搭档
        if need in seen:                 # 之前见过搭档 → 直接返回
            return [seen[need], i]
        seen[x] = i                      # 没见过 → 记下自己，供后面的数来找
    return []


print(two_sum([2, 7, 11, 15], 9))

[0, 1]


## 六、边界情况（面试必问）

算法题写完只是及格，**能主动说出边界情况**才是加分项：

| 情况 | 例子 | 为什么要注意 |
|---|---|---|
| 答案在末尾 | `[3,2,4], 6 → [1,2]` | 验证扫描到最后一个才命中 |
| 两个相同的数 | `[3,3], 6 → [0,1]` | 验证没有把同一个元素用两次 |
| 有负数 | `[-1,-2,-3,-4,-5], -8 → [2,4]` | 验证不是靠「大小关系」蒙的 |
| 找不到 | `[5], 10 → []` | 验证不会崩 |

In [5]:
assert two_sum([2, 7, 11, 15], 9) == [0, 1]
assert two_sum([3, 2, 4], 6) == [1, 2]
assert two_sum([3, 3], 6) == [0, 1]
assert two_sum([-1, -2, -3, -4, -5], -8) == [2, 4]
assert two_sum([5], 10) == []

print("全部测试通过 ✅")

全部测试通过 ✅


## 七、你的作业（现在做，大约 10 分钟）

**把上面的代码往上翻，看 2 分钟，然后翻回这里，合上眼睛自己写一遍。**

写不出来就再看一眼，再写。这个「看一眼 → 关掉 → 默写」的过程，才是真正记住的方式。

In [6]:
# ↓↓↓ 自己写，不要复制上面的 ↓↓↓
def my_two_sum(nums, target):
    # 在这里写你的代码
    pass


# 写完运行这一格验证
print(my_two_sum([2, 7, 11, 15], 9),    "期望 [0, 1]")
print(my_two_sum([3, 2, 4], 6),         "期望 [1, 2]")
print(my_two_sum([3, 3], 6),            "期望 [0, 1]")
print(my_two_sum([-1, -2, -3, -4, -5], -8), "期望 [2, 4]")

None 期望 [0, 1]
None 期望 [1, 2]
None 期望 [0, 1]
None 期望 [2, 4]


---

## 八、下一题（第 49 题：字母异位词分组）—— 只给思路，不给答案

**题目**：给一个字符串数组，把互为「字母异位词」的词分到一组。
异位词 = 字母相同、顺序不同。例如 `eat`、`tea`、`ate` 是一组。

**思路提示（就一句）**：

> 如果我能给每个词算出一个「签名」，让互为异位词的词**得到完全相同的签名**，
> 那么这就变成了最简单的「按签名分组」——又是一个字典。

**想一想**：什么样的签名能满足这个要求？（提示：想想 `eat` 和 `tea` 有什么关系，
对它们做同一个操作之后会不会变得一模一样。）

想不出来的话，再想这一层：**把单词的字母排序**，`eat` → `aet`，`tea` → `aet`。
两个完全一样的 key，就能用字典分到一组了。

关键词：`defaultdict(list)`、`sorted(word)`、`"".join(...)`。

自己先试，卡住了再问我——**我会告诉你我第一版会怎么写，但不会直接把答案贴给你。**
